# Forest Fire AI — Tabular Data Analysis
## Notebook 06: Exploratory Data Analysis of forestfires.csv


In [ ]:
import os, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
PLOTS    = IMPL / "artifacts" / "plots"
META_DIR = IMPL / "artifacts" / "metadata"
DATA_DIR = IMPL / "data" / "processed"
PLOTS.mkdir(parents=True, exist_ok=True)

CSV_PATH = ROOT / "forest+fires" / "forestfires.csv"
df = pd.read_csv(CSV_PATH)

print("Forest Fires CSV — EDA")
print("="*60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")


In [ ]:
print("\nData Types:")
print(df.dtypes.to_string())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"\nDescriptive Statistics:")
print(df.describe().round(3).to_string())


In [ ]:
# Add binary classification target
df['fire_occurred'] = (df['area'] > 0).astype(int)
df['area_log']      = np.log1p(df['area'])

print("Target Analysis — 'area' column (burned area in hectares):")
print(f"  Min:    {df['area'].min()}")
print(f"  Max:    {df['area'].max()}")
print(f"  Mean:   {df['area'].mean():.4f}")
print(f"  Median: {df['area'].median():.4f}")
print(f"  Zeros:  {(df['area'] == 0).sum()} ({100*(df['area']==0).mean():.1f}%)")
print(f"  >0:     {(df['area'] > 0).sum()} ({100*(df['area']>0).mean():.1f}%)")
print("\nBinary classification target (area > 0):")
print(df['fire_occurred'].value_counts().to_string())
print(f"\nTask decision: BOTH regression (area) + binary classification (fire_occurred)")


In [ ]:
# Encode categoricals
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
day_order   = ['mon','tue','wed','thu','fri','sat','sun']

df['month_num'] = df['month'].map({m: i for i, m in enumerate(month_order, 1)})
df['day_num']   = df['day'].map({d: i for i, d in enumerate(day_order, 1)})

# Numerical features
num_cols = ['X','Y','FFMC','DMC','DC','ISI','temp','RH','wind','rain',
            'month_num','day_num']

print("Numerical feature statistics:")
print(df[num_cols].describe().round(3).to_string())


In [ ]:
# Visualization — distributions
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle("Feature Distributions — Forest Fires Dataset", fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(df[col], bins=25, color='#2E8B57', edgecolor='black', alpha=0.8)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(PLOTS / "feature_distributions.png", dpi=100, bbox_inches='tight')
plt.close()
print("Feature distributions saved.")


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr_cols = num_cols + ['area', 'fire_occurred']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='RdYlGn',
            center=0, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS / "correlation_heatmap.png", dpi=100, bbox_inches='tight')
plt.close()

print("Top correlations with 'area':")
corr_with_area = corr['area'].drop('area').abs().sort_values(ascending=False)
print(corr_with_area.round(3).to_string())


In [ ]:
# Monthly fire distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

month_fires = df.groupby('month')['fire_occurred'].agg(['sum','count'])
month_fires = month_fires.reindex(month_order)
month_fires['pct'] = 100 * month_fires['sum'] / month_fires['count']

axes[0].bar(month_fires.index, month_fires['sum'], color='#E84040', edgecolor='black', alpha=0.8)
axes[0].set_title('Fire Events by Month', fontweight='bold')
axes[0].set_ylabel('Fire Events')
axes[0].set_xticklabels(month_fires.index, rotation=30)

area_by_month = df.groupby('month')['area'].sum().reindex(month_order)
axes[1].bar(area_by_month.index, area_by_month.values, color='#FF8C00', edgecolor='black', alpha=0.8)
axes[1].set_title('Total Burned Area by Month (ha)', fontweight='bold')
axes[1].set_ylabel('Total Area (ha)')
axes[1].set_xticklabels(area_by_month.index, rotation=30)

plt.tight_layout()
plt.savefig(PLOTS / "monthly_fire_analysis.png", dpi=100, bbox_inches='tight')
plt.close()
print("Monthly analysis saved.")


In [ ]:
# Save processed data for modeling
features = ['X','Y','FFMC','DMC','DC','ISI','temp','RH','wind','rain','month','day']
df_processed = df[features + ['area', 'fire_occurred', 'area_log']].copy()

df_processed.to_csv(DATA_DIR / "forestfires_processed.csv", index=False)

tab_meta = {
    'features': features,
    'numerical_features': ['X','Y','FFMC','DMC','DC','ISI','temp','RH','wind','rain'],
    'categorical_features': ['month','day'],
    'regression_target': 'area',
    'classification_target': 'fire_occurred',
    'n_samples': len(df),
    'fire_positive': int((df['fire_occurred']==1).sum()),
    'fire_negative': int((df['fire_occurred']==0).sum())
}
with open(META_DIR / "tabular_metadata.json", "w") as f:
    json.dump(tab_meta, f, indent=2)

print(f"Processed data saved: {DATA_DIR}/forestfires_processed.csv")
print(f"Tabular metadata saved: {META_DIR}/tabular_metadata.json")
print("\nNotebook 06 complete.")
